# **IRIS Agents**

In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')
iris_toolkit = Toolkit(name='IRIS', url = 'http://localhost:9002/mcp')

iris_toolkit

Toolkit(name='IRIS', url='http://localhost:9002/mcp')

### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name='Agent007', text='You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name='Agent007', text='Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt('Agent007')
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production(name='AgentSpace', agents=[molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 05/04/2026 13:23:48
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 05/04/2026 13:23:48
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 05/04/2026 13:23:48
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 05/04/2026 13:23:48
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 05/04/2026 13:23:49
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

'Here are great summer hiking options around Boston (about an hour or less):\n- Blue Hills Reservation (Milton/Quincy): Skyline Trail 9–10 mi strenuous with views; plenty of easier loops; Ponkapoag Bog boardwalk.\n- Middlesex Fells (Medford/Stoneham/Winchester): Skyline 7–8 mi moderate; Rock Circuit ~4 mi rocky; some access via MBTA.\n- Breakheart Reservation (Saugus/Wakefield): 3–5 mi around lakes; mix of paved/dirt; family-friendly.\n- Lynn Woods Reservation (Lynn): Stone Tower and Dungeon Rock; extensive rocky singletrack.\n- Walden Pond + Hapgood Wright Town Forest (Concord): 1.7 mi pond loop; connect to Emerson–Thoreau Amble; swimming after.\n- Minute Man NHP, Battle Road Trail (Lexington–Concord): ~5 mi one-way; flat; historic sites.\n- Great Meadows NWR (Concord): easy levee/boardwalk-style paths; excellent birding (buggy in summer).\n- World’s End (Hingham, Trustees): 3–4 mi carriage roads with harbor views; entry fee; breezy.\n- Crane Beach & Castle Neck (Ipswich, Trustees): b

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

True

Agents can be configured with a default reasoning effort and response format, but these parameters can be overridden at runtime.

In [14]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             reasoning_effort='low',
             toolkits=[utils_toolkit, iris_toolkit],
             response_format=MollyResponse)


Load started on 05/04/2026 13:24:20
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 05/04/2026 13:24:22
Loading file Agents.Message.MollyResponse.cls as udl
Compiling class Agents.Message.MollyResponse
Compiling table Agents_Message.MollyResponse
Compiling routine Agents.Message.MollyResponse.1
Load finished successfully.


In [15]:
Production(name='AgentSpace', agents=[molly, alex]).start()


Load started on 05/04/2026 13:24:23
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 05/04/2026 13:24:23
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 05/04/2026 13:24:24
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 05/04/2026 13:24:24
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 05/04/2026 13:24:24
Loading file Agents.Message.LLMOutput.cls as udl
Compiling class Agents.Message.LLMOutp

Without any context, the agent does not construct its memory from the database, nor does it log the response, operating in a stateless manner. Model usage is still logged.

In [16]:
molly('Which tables do we have in IRIS in the Agents namespace?')

'{"text": "Agents namespace tables: SQLUser.TestModel", "reasoning": "Used IRIS list_tables for the Agents namespace and read the returned list."}'

Agents can be directly invoked to use a tool using compatible parameters

In [17]:
molly.use(toolkit='IRIS', tool='query', params={'namespace':'Agents', 'sql': 'Select * from Agents.Agent'})

[{'agent_name': 'Alex',
  'description': 'Test Agent 1',
  'system_prompt_id': 'alex_system',
  'model': 'gpt-5',
  'response_format': 'AlexResponse',
  'reasoning_effort': 'medium',
  'persist_reasoning': '1'},
 {'agent_name': 'Molly',
  'description': 'Test Agent 2',
  'system_prompt_id': 'molly_system',
  'model': 'gpt-5',
  'response_format': 'MollyResponse',
  'reasoning_effort': 'low',
  'persist_reasoning': '1'}]

When context is specified, the agent constructs its conversational context from database before querying the LLM

In [18]:
molly(message='What is the weather today?', chat=context)

'{"text": "Today in Washington, DC: Cloudy with a high of 26\\u00b0 and a low of 13\\u00b0.", "reasoning": "Used the provided weather tool result for Washington, DC."}'

Here, we override the agent's default reasoning effort at runtime

In [19]:
molly(message='Recommend some good food spots for lunch', chat='travel', reasoning_effort='high')

'{"text": "Here are solid lunch spots in Washington, DC (mix of quick and sit-down):\\n\\nQuick & casual:\\n- RASA (Indian bowls; bright flavors, gluten-free friendly)\\n- Shouk (plant-based Israeli street food)\\n- Little Sesame (hummus bowls, pita sandwiches)\\n- A Baked Joint (hearty sandwiches, coffee)\\n- Call Your Mother (bagel sandwiches; expect a short line)\\n- Wiseguy Pizza (big New York\\u2013style slices; fast)\\n\\nSit-down options:\\n- Zaytinya (Jose Andr\\u00e9s\\u2019 Mediterranean mezze; great for sharing)\\n- Oyamel (creative Mexican; tacos and small plates)\\n- Unconventional Diner (modern comfort food near Convention Center)\\n- Old Ebbitt Grill (classic DC institution near the White House)\\n- The Hamilton (broad American menu; good for groups, downtown)\\n- Daikaya (ramen shop in Chinatown)\\n\\nIf you\\u2019re by the National Mall museums:\\n- Mitsitam Native Foods Caf\\u00e9 (Smithsonian Museum of the American Indian)\\n- Sweet Home Caf\\u00e9 (National Museum o

Here we override the agent's default response format at runtime

In [20]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 05/04/2026 13:26:01
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 05/04/2026 13:26:01
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "Centrolina", "cuisine": "Italian"}, {"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Osteria Morini", "cuisine": "Italian"}, {"name": "RPM Italian", "cuisine": "Italian"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "2Amys", "cuisine": "Italian"}, {"name": "Anju", "cuisine": "Korean"}, {"name": "Daikaya", "cuisine": "Japanese"}, {"name": "Sushi Taro", "cuisine": "Japanese"}, {"name": "Baan Siam", "cuisine": "Thai"}, {"name": "Tiger Fork", "cuisine": "Cantonese"}, {"name": "Maketto", "cuisine": "Taiwanese/Cambodian"}], "reasoning": "Curated Washington, DC lunch options that match Italian and Asian preferences, mixing casual and sit-down spots."}'

Usage information can be fetched using the API for each Agent, Production or Chat. Productions can also be filtered by certain agents.

In [21]:
Chat('travel').usage()

{'input_tokens': 37520,
 'output_tokens': 51037,
 'output_reasoning_tokens': 43136,
 'total_tokens': 88557}

In [22]:
Production('AgentSpace').usage()

{'input_tokens': 69988,
 'output_tokens': 88264,
 'output_reasoning_tokens': 69760,
 'total_tokens': 158252}

In [23]:
molly.usage()

{'input_tokens': 69988,
 'output_tokens': 88264,
 'output_reasoning_tokens': 69760,
 'total_tokens': 158252}

In [24]:
Production('AgentSpace').usage(agents=[Agent('Molly')])

{'input_tokens': 69988,
 'output_tokens': 88264,
 'output_reasoning_tokens': 69760,
 'total_tokens': 158252}

Productions and Agents can be deleted using the `delete()` methods. This operation removes the Objectscript classes backing these objects as well.

In [25]:
Production('AgentSpace').delete()

Deleted production: User.AgentSpace

Deleting class Agents.REST.Dispatch.AgentSpaceCleaned up production-owned artifacts for: AgentSpace


In [26]:
Agent('Molly').delete()
try:
    molly('Hello')
except KeyError as e:
    print(e)


Deleting class Agents.Gateway.MollyService
Deleting class Agents.Process.Molly"No Agent found for 'Molly'"
